## 1. Carregamento da base convertida

Nesta etapa, carregamos os arquivos derivados do Amazon Delivery Dataset já convertidos para o formato do projeto. O objetivo e validar se a base principal e a tabela de veículos estao acessiveis e estruturadas corretamente.

In [4]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path("..").resolve()
DATASET_DIR = BASE_DIR / "dataset" / "converted"

stops_path = DATASET_DIR / "stops_from_amazon.csv"
vehicles_path = DATASET_DIR / "vehicles_from_amazon.csv"

stops_df = pd.read_csv(stops_path)
vehicles_df = pd.read_csv(vehicles_path)

print("Stops shape:", stops_df.shape)
print("Vehicles shape:", vehicles_df.shape)

display(stops_df.head())
display(vehicles_df.head())

Stops shape: (18936, 7)
Vehicles shape: (4, 3)


,stop_id,name,x,y,demand,priority,is_depot
0,DEPOT-STORE,Store Depot,11.003669,76.976494,0,0,True
1,rjto796129700,médicamentos_críticos,11.053669,77.026494,2,5,False
2,vmau710398846,médicamentos_críticos,10.043064,76.347589,2,5,False
3,zyvo118176215,insumos_médicos,19.266269,72.926721,3,5,False
4,uhfs888375680,médicamentos_críticos,12.351072,76.694878,2,5,False


,vehicle_id,capacity,max_distance
0,motorcycle,12,25
1,van,30,40
2,scooter,10,25
3,bicycle,6,15


## 2. Inspecao inicial da base

Aqui verificamos o volume de paradas, a quantidade de veículos disponíveis, a distribuição entre tipos de entrega e a estrutura de prioridades. Isso ajuda a entender se o recorte médico ficou coerente com a narrativa do projeto.

In [5]:
print("Quantidade total de paradas:", len(stops_df))
print("Quantidade de veículos:", len(vehicles_df))
print()

print("Distribuicao por tipo de entrega:")
display(stops_df["name"].value_counts().to_frame("count"))

print("Distribuicao de prioridade:")
display(stops_df["priority"].value_counts().sort_index().to_frame("count"))

print("Depósito único?:", stops_df["is_depot"].sum() == 1)
print("Quantidade de entregas sem depósito:", (~stops_df["is_depot"]).sum())

Quantidade total de paradas: 18936
Quantidade de veículos: 4

Distribuicao por tipo de entrega:


,count
name,
insumos_médicos,10795
médicamentos_críticos,8140
Store Depot,1


Distribuicao de prioridade:


,count
priority,
0,1
4,4900
5,14035


Depósito único?: True
Quantidade de entregas sem depósito: 18935


## 3. Qualidade estrutural dos dados

Nesta etapa avaliamos valores ausentes, duplicidades e estatisticas descritivas da base. O objetivo e identificar problemas que possam distorcer os experimentos do algoritmo genético.

In [6]:
entregas_df = stops_df[~stops_df["is_depot"]].copy()

print("Entregas:", len(entregas_df))
print()

print("Valores ausentes:")
display(entregas_df.isna().sum().to_frame("missing"))

print("Duplicados por stop_id:", entregas_df["stop_id"].duplicated().sum())
print("Duplicados por coordenadas (x, y):", entregas_df.duplicated(subset=["x", "y"]).sum())
print()

print("Resumo numerico:")
display(entregas_df[["x", "y", "demand", "priority"]].describe())

Entregas: 18935

Valores ausentes:


,missing
stop_id,0
name,0
x,0
y,0
demand,0
priority,0
is_depot,0


Duplicados por stop_id: 0
Duplicados por coordenadas (x, y): 14939

Resumo numerico:


,x,y,demand,priority
count,18935.000000,18935.000000,18935.000000,18935.000000
mean,17.480404,70.904509,2.570108,4.741220
std,7.326526,20.986076,0.495074,0.437976
min,0.010000,0.010000,2.000000,4.000000
25%,12.985996,73.280937,2.000000,4.000000
50%,18.642718,75.997648,3.000000,5.000000
75%,22.785747,78.084159,3.000000,5.000000
max,31.054057,88.563452,3.000000,5.000000


## 4. Definicao da amostra de trabalho

Como a base completa possui muitas entregas, criamos uma amostra controlada para os experimentos dos notebooks. A amostra preserva a proporcao entre médicamentos críticos e insumos médicos, mantendo reprodutibilidade com `random_state` fixo.

In [8]:
random_staté = 42

médicamentos_df = entregas_df[entregas_df["name"] == "médicamentos_críticos"].sample(
    n=43, random_state=random_state
)

insumos_df = entregas_df[entregas_df["name"] == "insumos_médicos"].sample(
    n=57, random_state=random_state
)

sample_entregas_df = pd.concat(
    [médicamentos_df, insumos_df],
    ignore_index=True
)

sample_stops_df = pd.concat(
    [stops_df[stops_df["is_depot"]], sample_entregas_df],
    ignore_index=True
)

print("Amostra final de paradas:", sample_stops_df.shape)
display(sample_stops_df["name"].value_counts(dropna=False).to_frame("count"))
display(sample_stops_df["priority"].value_counts().sort_index().to_frame("count"))
display(sample_stops_df.head())

Amostra final de paradas: (101, 7)


,count
name,
insumos_médicos,57
médicamentos_críticos,43
Store Depot,1


,count
priority,
0,1
4,26
5,74


,stop_id,name,x,y,demand,priority,is_depot
0,DEPOT-STORE,Store Depot,11.003669,76.976494,0,0,True
1,cobx423154877,médicamentos_críticos,17.501028,78.419645,2,5,False
2,gbbd486976829,médicamentos_críticos,22.340526,73.200937,2,5,False
3,qenp660643292,médicamentos_críticos,0.040000,0.040000,2,5,False
4,mpmj126993214,médicamentos_críticos,13.023041,77.793237,2,5,False


## 5. Persistencia da amostra

Nesta etapa salvamos a amostra selecionada em arquivos proprios para reutilizacao nos próximos notebooks. Isso evita recomputacao desnecessaria e garante consistencia entre os experimentos.

In [9]:
OUTPUT_DIR = BASE_DIR / "dataset" / "samples"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sample_stops_path = OUTPUT_DIR / "stops_sample_100.csv"
sample_vehicles_path = OUTPUT_DIR / "vehicles_sample_100.csv"

sample_stops_df.to_csv(sample_stops_path, index=False)
vehicles_df.to_csv(sample_vehicles_path, index=False)

print("Arquivo de paradas salvo em:", sample_stops_path)
print("Arquivo de veículos salvo em:", sample_vehicles_path)

Arquivo de paradas salvo em: /Volumes/SSD-EXT-MAC-FB/AI-Postgraduate/Tech Challenge B/Entregavel/fase-02/tech-challenge/dataset/samples/stops_sample_100.csv
Arquivo de veículos salvo em: /Volumes/SSD-EXT-MAC-FB/AI-Postgraduate/Tech Challenge B/Entregavel/fase-02/tech-challenge/dataset/samples/vehicles_sample_100.csv


## 6. Inspecao de coordenadas suspeitas

Aqui verificamos a existencia de coordenadas muito próximas de zero, pois esses registros podem representar anomalias ou valores pouco confiaveis. Se mantidos, eles podem distorcer o calculo de distâncias e a qualidade das rotas.

In [10]:
suspeitos_df = sample_entregas_df[
    (sample_entregas_df["x"] <= 1) | (sample_entregas_df["y"] <= 1)
].copy()

print("Quantidade de coordenadas suspeitas:", len(suspeitos_df))
display(suspeitos_df[["stop_id", "name", "x", "y", "demand", "priority"]].sort_values(["x", "y"]))

Quantidade de coordenadas suspeitas: 12


,stop_id,name,x,y,demand,priority
22,rvpq631939731,médicamentos_críticos,0.01,0.01,2,5
30,xojy003718814,médicamentos_críticos,0.02,0.02,2,5
55,cxqw534964654,insumos_médicos,0.02,0.02,3,4
56,ttzq314291900,insumos_médicos,0.02,0.02,3,4
27,dyqn470935235,médicamentos_críticos,0.03,0.03,2,5
80,szwa468735133,insumos_médicos,0.03,0.03,3,4
97,nwso874502737,insumos_médicos,0.03,0.03,3,5
2,qenp660643292,médicamentos_críticos,0.04,0.04,2,5
92,hvrd695156825,insumos_médicos,0.04,0.04,3,4
68,gjxy242988280,insumos_médicos,0.07,0.07,3,5


## 7. Limpeza de coordenadas anômalas

A amostra inicial continha entregas com coordenadas muito próximas de zero, o que pode distorcer fortemente o calculo de distâncias. Nesta etapa removemos esses pontos e recompomos a amostra para manter 100 entregas com a mesma proporcao entre as classes.

In [11]:
random_staté = 42

amostra_limpa_df = sample_entregas_df[
    (sample_entregas_df["x"] > 1) & (sample_entregas_df["y"] > 1)
].copy()

faltantes = 100 - len(amostra_limpa_df)
print("Entregas após limpeza:", len(amostra_limpa_df))
print("Faltantes para completar 100:", faltantes)

pool_reposicao_df = entregas_df[
    (entregas_df["x"] > 1) & (entregas_df["y"] > 1)
].copy()

pool_reposicao_df = pool_reposicao_df[
    ~pool_reposicao_df["stop_id"].isin(amostra_limpa_df["stop_id"])
]

reposicao_med_df = pool_reposicao_df[
    pool_reposicao_df["name"] == "médicamentos_críticos"
].sample(n=4, random_state=random_state)

reposicao_ins_df = pool_reposicao_df[
    pool_reposicao_df["name"] == "insumos_médicos"
].sample(n=8, random_state=random_state)

sample_entregas_df = pd.concat(
    [amostra_limpa_df, reposicao_med_df, reposicao_ins_df],
    ignore_index=True
)

sample_stops_df = pd.concat(
    [stops_df[stops_df["is_depot"]], sample_entregas_df],
    ignore_index=True
)

print("Amostra final após limpeza:", sample_stops_df.shape)
display(sample_stops_df["name"].value_counts().to_frame("count"))
display(sample_stops_df[["x", "y"]].describe())

Entregas após limpeza: 88
Faltantes para completar 100: 12
Amostra final após limpeza: (101, 7)


,count
name,
insumos_médicos,57
médicamentos_críticos,43
Store Depot,1


,x,y
count,101.000000,101.000000
mean,19.061474,76.357734
std,5.594196,3.228675
min,10.107014,72.798778
25%,13.050324,73.878172
50%,19.201458,75.927333
75%,22.805835,77.694130
max,30.965204,88.453187


## 8. Salvamento da amostra final

Depois da limpeza, salvamos novamente a amostra definitiva que será usada nos próximos notebooks. A partir deste ponto, esta passa a ser a base oficial dos experimentos.

In [12]:
sample_stops_df.to_csv(sample_stops_path, index=False)
vehicles_df.to_csv(sample_vehicles_path, index=False)

print("Amostra limpa salva em:", sample_stops_path)
print("Tabela de veículos salva em:", sample_vehicles_path)

Amostra limpa salva em: /Volumes/SSD-EXT-MAC-FB/AI-Postgraduate/Tech Challenge B/Entregavel/fase-02/tech-challenge/dataset/samples/stops_sample_100.csv
Tabela de veículos salva em: /Volumes/SSD-EXT-MAC-FB/AI-Postgraduate/Tech Challenge B/Entregavel/fase-02/tech-challenge/dataset/samples/vehicles_sample_100.csv


## 9. Sintese da etapa

A base principal do projeto foi definida a partir do Amazon Delivery Dataset convertido para o formato de roteirizacao. Após o recorte médico, a base passou a conter entregas classificadas em `médicamentos_críticos` e `insumos_médicos`.

Para viabilizar os experimentos com algoritmo genético, foi criada uma amostra reprodutível de 100 entregas, preservando a proporcao entre as duas classes. Tambem foram removidas coordenadas anômalas muito próximas de zero, reduzindo o risco de distorcoes no calculo das distâncias.

Com isso, o projeto passa a contar com uma base limpa, controlada e adequada para os próximos notebooks de modelagem genética, avaliação e geracao de relatorios.